In [22]:
import numpy as np
from pyscf import gto, scf, cc
import jax
jax.config.update("jax_enable_x64", True)

d = 100

natom = 1
atoms = ""
for n in range(natom):
    shift = n*d
    atoms += f'N {0.0+shift} 0.0 0.0 \n'
    atoms += f'N {0.0+shift} 0.0 2.4 \n'

spin = 0
mol = gto.M(atom=atoms, 
            basis="ccpvqz", 
            spin=spin, 
            unit='B',
            verbose=4)
mol.build()

mf = scf.RHF(mol)
mf.kernel()
    
stable = False
while not stable:
    print(f'mean-field stability test')
    if not stable:
        mo_i, _, stable,_ = mf.stability(return_status=True)
        dm = mf.make_rdm1(mo_i,mf.mo_occ)
        mf.kernel(dm0=dm)
    elif stable:
        print(f'UHF Energy: {mf.e_tot}, stability {stable}')
        break


mycc = cc.CCSD(mf)
mycc.set_frozen()
mycc.kernel()

print(mycc.energy(mycc.t1, mycc.t2*0))

System: uname_result(system='Linux', node='sharmagroup-rn', release='7.0.0-28-generic', version='#28~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Wed Jul  1 15:50:57 UTC 2', machine='x86_64')  Threads 16
Python 3.12.13 | packaged by Anaconda, Inc. | (main, Mar 19 2026, 20:20:58) [GCC 14.3.0]
numpy 2.4.4  scipy 1.17.1  h5py 3.16.0
Date: Wed Aug  5 14:37:19 2026
PySCF version 2.12.1
PySCF path  /home/sharmagroup/sharmagroup/pyscf
GIT ORIG_HEAD 3d1768f5e33b144b606c3d2c81c12ee54d794501
GIT HEAD (branch master) f0861da51f017364d8bbaa20b742a94f3733305f

[ENV] OLD_PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:
[ENV] PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:/home/sharmagroup/sharmagroup/pyscf-forge:
[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 2
[INPUT] num. electrons = 14
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = B
[INPUT] Symbol           X                Y                Z      

In [23]:
import time
import numpy as np
from jax import numpy as jnp
from jax import jit
import opt_einsum as oe

from afqmc import config
from afqmc import prep

from functools import partial
print = partial(print, flush=True)
config.setup_jax()

Hostname:     sharmagroup-rn
System:       Linux
Node:         sharmagroup-rn
Release:      7.0.0-28-generic
Machine:      x86_64
Processor:    x86_64
JAX backend:  GPU
JAX devices:  [CudaDevice(id=0)]
Device kind:  NVIDIA GeForce RTX 5060 Ti
Platform:     gpu


In [24]:
from afqmc import integral
integral.prep_integral(mycc, chol_cut=1e-6)


Preparing AFQMC calculation
CCSD type input object
Calculating Cholesky integrals
Cholesky shape: (807, 108, 108) 
Finished calculating Cholesky integrals
Size of the correlation space:
Number of electrons:        [5, 5]
Number of basis functions:  108
Number of Cholesky vectors: 807


In [25]:
options = {'eql_time': 10,
           'n_blocks': 100,
           'n_walkers': 10,
           'mix_precision': False,
           'seed': 17,
           'guide': 'rhf',
           'trial': 'rpt2ccsd_bar',
           }

In [26]:
ham_data, ham, prop, trial, wave_data, sampler, options = prep.init_afqmc(options=options)
wave_data["rdm1"] = trial.get_rdm1(wave_data)
ham_data = ham.build_measurement_intermediates(ham_data, trial, wave_data)
ham_data = ham.build_propagation_intermediates(ham_data, prop, trial, wave_data)
prop_data = prep.init_hf_prop_data(trial, wave_data, ham_data, options)
print(mf.e_tot - prop_data["e_estimate"])


QMC Parameters
eql_time        -         10
n_blocks        -        100
n_walkers       -         10
mix_precision   -      False
seed            -         17
guide           -        rhf
trial           - rpt2ccsd_bar
dt              -      0.005
n_exp_terms     -          6
n_prop_steps    -         50
walker_type     -        rhf
n_batch         -          1
max_error       -          0
nchol_chunk     -        100
max_memory      -       2000
free_projection -      False

Load system from Integral File
Maximum memory per walker:            200.00 MB
Maximum number of Cholesky per chunk: 1123
Number of Cholesky chunks:            1
Number of Cholesky per chunk:         807
Number of padding Cholesky:           0

QMC System
Number of electrons: (5, 5)
Spin Multiplicity:   0
Number of orbitals:  108
Number of Chol:      807

Initalize QMC walkers by HF
2.1469887201419624e-08


In [27]:
def pt2_energy_formula(h0, t2, e0, e1):
    return h0 + e0 + e1 - t2*e0

In [28]:
norb = trial.norb
h0 = ham_data["h0"]
h1 = ham_data["h1"][0]
chol = ham_data["chol"].reshape(-1, norb, norb)

In [29]:
walker_init = prop_data['walkers'][0]
obar, t2, e0, e1 = \
    trial._calc_energy_pt(walker_init, ham_data, wave_data)
print(pt2_energy_formula(h0, t2, e0, e1)-mycc.e_tot)


(1.6182070794457104e-08+0j)


In [76]:
from jax import jit, lax
import opt_einsum as oe

@partial(jit, static_argnums=0)
def _calc_energy_pt2_decomposed(self, walker, ham_data, wave_data):
    # becareful tau is complex!

    if self.mix_precision:
        rtype = jnp.float32
        ctype = jnp.complex64
    else:
        rtype = jnp.float64
        ctype = jnp.complex128
    
    nocc, norb = self.nelec[0], self.norb
    nchol_chunk = self.nchol_chunk  # nchol per chunk

    tau = wave_data["tau"]
    h1 = ham_data["h1_bar"]
    chol = ham_data["chol_bar"]
    walker_bar = wave_data['exp_t1'] @ walker

    obar = jnp.linalg.det(walker_bar[:walker_bar.shape[1], :]) ** 2

    green = (walker_bar.dot(jnp.linalg.inv(walker_bar[: walker_bar.shape[1], :]))).T
    green_occ = green[:, nocc:]
    greenp = jnp.vstack((green_occ, -jnp.eye(norb - nocc)))
    green_ov = green[:nocc,nocc:]

    rot_chol = chol[:, :nocc, :]
    nchol = chol.shape[0]
    # chunk_size = naux // nchol_chunk

    # 1 body energy
    hg = oe.contract("pi,pi->", h1[:nocc, :], green, backend="jax")
    e1_0 = 2 * hg

    def scan_tau1(carry, x):
        tau_y = x
        taug_y = oe.contract("ia,ja->ij", tau_y, green_ov, backend="jax")
        taugp_y = oe.contract("jb,pb->jp", tau_y, greenp, backend="jax")
        taugpg_y = oe.contract("jp,jq->pq", taugp_y, green[:nocc,:], backend="jax")
        taugg_y = oe.contract("ji,jq->iq", taug_y, green[:nocc,:], backend="jax")
        tr_taug_y = oe.contract("ii->", taug_y, backend="jax")
        # t2o_c = oe.contract("y,y->", tr_taug, tr_taug, backend="jax")
        carry[0] += tr_taug_y ** 2
        carry[1] += oe.contract("ij,ji->", taug_y, taug_y, backend="jax")

        t2_green_c = tr_taug_y * taugpg_y
        t2_green_e = oe.contract("ip,iq->pq", taugp_y, taugg_y, backend="jax")
        carry[2] += 2 * t2_green_c - t2_green_e

        return carry, 0.0
    
    init = [jnp.zeros((), jnp.complex128),                 # t2o_c   (scalar)
            jnp.zeros((), jnp.complex128),                 # t2o_e   (scalar)
            jnp.zeros((norb, norb), jnp.complex128)]       # t2_green (matrix)

    [t2o_c, t2o_e, t2_green], _ = lax.scan(scan_tau1, init, tau)

    t2o = 2 * t2o_c - t2o_e # <HF|T2|walker>

    e1_2_1 = t2o * e1_0

    e1_2_2 = -2 * oe.contract("pq,pq->", h1, t2_green, backend="jax")
    e1_2 = e1_2_1 + e1_2_2 # <HF|T2 h1|walker>/<HF|walker>

    # pad with zero cholesky vectors — contributes nothing to any contraction
    npad = (-nchol) % nchol_chunk
    chol = jnp.concatenate([chol, jnp.zeros((npad, norb, norb))], axis=0)
    rot_chol = jnp.concatenate([rot_chol, jnp.zeros((npad, nocc, norb))], axis=0)

    # reshape into chunks: (n_chunks, chunk_size, ...)
    nchunk = (nchol + npad) // nchol_chunk
    chol = chol.reshape(nchunk, nchol_chunk, norb, norb)
    rot_chol = rot_chol.reshape(nchunk, nchol_chunk, nocc, norb)

    # two body — scan over chunks, explicit contractions within a chunk
    def scan_chunk(carry, x):
        chol_c, rot_chol_c = x  # (chunk_size, norb, norb), (chunk_size, nocc, norb)

        gl = oe.contract("ir,gqr->giq", green, chol_c, backend="jax")
        gl_c = oe.contract("gii->g", gl[:, :, :nocc], backend="jax")
        e2_0_c = oe.contract("g,g->", gl_c, gl_c, backend="jax") * 2
        e2_0_e = -oe.contract("gij,gji->", gl[:, :, :nocc], gl[:, :, :nocc], backend="jax")
        carry[0] += e2_0_c + e2_0_e

        lt2g = oe.contract("gpr,pr->g", 
                            chol_c.astype(rtype), 
                            t2_green.astype(ctype), 
                            backend="jax")
        carry[1] += -oe.contract("g,g->", 
                                    lt2g.astype(ctype), 
                                    gl_c.astype(ctype), 
                                    backend="jax")

        lt2_green = oe.contract("gir,qr->giq", 
                                rot_chol_c.astype(rtype), 
                                t2_green.astype(ctype), 
                                backend="jax")
        # t_iajb |G_ia G_js Gp_pb| G_qr L_pr L_qs
        carry[2] += 0.5 * oe.contract("giq,giq->", 
                                        gl.astype(ctype), 
                                        lt2_green.astype(ctype), 
                                        backend="jax")

        # t_iajb G_ir G_js Gp_pa Gp_qb L_pr L_qs type
        glgp = oe.contract("gir,rb->gib", 
                            gl.astype(ctype), 
                            greenp.astype(ctype), 
                            backend="jax")
        
        def scan_tau2(carry, x):
            tau_y = x
            tauglgp_y = oe.contract("ia,gja->gij", 
                                tau_y.astype(ctype),
                                glgp.astype(ctype), 
                                backend="jax")
            tr_tauglgp_y = oe.contract("gii->g", 
                                tauglgp_y.astype(ctype),
                                backend="jax")
            carry[0] += oe.contract("g,g->", 
                                tr_tauglgp_y.astype(ctype), 
                                tr_tauglgp_y.astype(ctype), 
                                backend="jax").astype(jnp.complex128)
            carry[1] += oe.contract("gij,gji->", 
                                tauglgp_y.astype(ctype), 
                                tauglgp_y.astype(ctype), 
                                backend="jax").astype(jnp.complex128)
            return carry, 0.0
        
        init = [jnp.zeros((), jnp.complex128),                
                jnp.zeros((), jnp.complex128)]       

        [l2t2_c, l2t2_e], _ = lax.scan(scan_tau2, init, tau)

        carry[3] += (2*l2t2_c - l2t2_e).astype(jnp.complex128)

        return carry, 0.0

    [e2_0, e2_2_2_1, e2_2_2_2, e2_2_3], _ = lax.scan(
        scan_chunk, [0.0, 0.0, 0.0, 0.0], (chol, rot_chol)
    )

    e2_2_1 = e2_0 * t2o
    e2_2_2 = 4 * (e2_2_2_1 + e2_2_2_2)
    e2_2 = e2_2_1 + e2_2_2 + e2_2_3

    e0 = e1_0 + e2_0  # <psi|(h1+h2)|phi>/<psi|phi>
    e1 = e1_2 + e2_2  # <psi|t2(h1+h2)|phi>/<psi|phi>

    return obar, t2o, e0, e1

In [31]:
# @partial(jit, static_argnums=0)
# def _calc_energy_pt2_decomposed_new(self, walker, ham_data, wave_data):
#     # NOTE: tau is complex
#     if self.mix_precision:
#         rtype, ctype = jnp.float32, jnp.complex64
#     else:
#         rtype, ctype = jnp.float64, jnp.complex128

#     nocc, norb = self.nelec[0], self.norb
#     nvir = norb - nocc
#     nchol_chunk = self.nchol_chunk
#     tau  = wave_data["tau"]          # (ny, nocc, nvir), complex
#     h1   = ham_data["h1_bar"]
#     chol = ham_data["chol_bar"]      # (nchol, norb, norb)

#     walker_bar = wave_data['exp_t1'] @ walker
#     obar  = jnp.linalg.det(walker_bar[:walker_bar.shape[1], :]) ** 2
#     green = (walker_bar.dot(jnp.linalg.inv(walker_bar[:walker_bar.shape[1], :]))).T  # (nocc, norb)
#     green_occ = green[:, nocc:]
#     greenp    = jnp.vstack((green_occ, -jnp.eye(nvir)))   # (norb, nvir)
#     green_ov  = green[:nocc, nocc:]                       # (nocc, nvir)
#     green_o   = green[:nocc, :]                           # (nocc, norb)  (== green)
#     rot_chol  = chol[:, :nocc, :]                         # (nchol, nocc, norb)
#     nchol = chol.shape[0]

#     # ---- one-body mean field ----
#     hg = oe.contract("pi,pi->", h1[:nocc, :], green, backend="jax")
#     e1_0 = 2 * hg

#     # ---- pad + chunk cholesky ----
#     npad = (-nchol) % nchol_chunk
#     chol   = jnp.concatenate([chol,     jnp.zeros((npad, norb, norb), chol.dtype)], axis=0)
#     rot_chol = jnp.concatenate([rot_chol, jnp.zeros((npad, nocc, norb), rot_chol.dtype)], axis=0)
#     nchunk = (nchol + npad) // nchol_chunk
#     chol     = chol.reshape(nchunk, nchol_chunk, norb, norb)
#     rot_chol = rot_chol.reshape(nchunk, nchol_chunk, nocc, norb)

#     # ---- Cholesky scan: everything with NO y dependence ----
#     # e2_0        : mean-field two-body
#     # CG, GR      : operators so the y-dependent e2_2_2 can be built in the y-scan
#     # glgp (out)  : gathered per-cholesky, needed by the y-dependent e2_2_3
#     def chol_step(carry, x):
#         e2_0, CG, GR = carry
#         chol_c, rot_chol_c = x                                   # (gc,n,n), (gc,o,n)
#         gl     = oe.contract("ir,gqr->giq", green, chol_c, backend="jax")   # (gc,o,n)
#         gl_occ = gl[:, :, :nocc]                                            # (gc,o,o)
#         gl_c   = oe.contract("gii->g", gl_occ, backend="jax")              # (gc,)
#         e2_0 = e2_0 + 2 * oe.contract("g,g->", gl_c, gl_c, backend="jax") \
#                         - oe.contract("gij,gji->", gl_occ, gl_occ, backend="jax")
#         # CG[p,r] = sum_g gl_c[g] chol[g,p,r]           -> for e2_2_2_1
#         CG = CG + oe.contract("g,gpr->pr", gl_c, chol_c, backend="jax")     # (n,n)
#         # GR[i,q,r] = sum_g gl[g,i,q] rot_chol[g,i,r]   -> for e2_2_2_2
#         GR = GR + oe.contract("giq,gir->iqr", gl, rot_chol_c, backend="jax")  # (o,n,n)
#         glgp = oe.contract("gir,rb->gib", gl, greenp, backend="jax")        # (gc,o,v)
#         return (e2_0, CG, GR), glgp

#     (e2_0, CG, GR), glgp = lax.scan(
#         chol_step,
#         (jnp.zeros((), ctype),
#          jnp.zeros((norb, norb), ctype),
#          jnp.zeros((nocc, norb, norb), ctype)),
#         (chol, rot_chol),
#     )
#     glgp = glgp.reshape(nchunk * nchol_chunk, nocc, nvir).astype(ctype)  # (G+pad, o, v)
#     CG = CG.astype(ctype)
#     GR = GR.astype(ctype)

#     # ---- final scan over the decomposition rank y ----
#     # each step sees ONE tau vector t[i,a] (no y axis inside)
#     def y_step(carry, t):                                 # t: (nocc, nvir)
#         t2o, e1_2_2, e2_2_2, e2_2_3 = carry
#         t = t.astype(ctype)

#         # --- one-body pieces for this y ---
#         taug_y  = oe.contract("ia,ja->ij", t, green_ov, backend="jax")      # (o,o)
#         tr_y    = oe.contract("ii->", taug_y, backend="jax")               # scalar
#         taugp_y = oe.contract("jb,pb->jp", t, greenp, backend="jax")        # (o,n)
#         taugg_y = oe.contract("ji,jq->iq", taug_y, green_o, backend="jax")  # (o,n)
#         t2gc    = oe.contract("jp,jq->pq", taugp_y, green_o, backend="jax") # (n,n)
#         t2_green_y = 2 * (tr_y * t2gc) \
#                      - oe.contract("ip,iq->pq", taugp_y, taugg_y, backend="jax")   # (n,n)

#         t2o    = t2o + 2 * tr_y ** 2 \
#                      - oe.contract("ij,ji->", taug_y, taug_y, backend="jax")
#         e1_2_2 = e1_2_2 - 2 * oe.contract("pq,pq->", h1.astype(ctype), t2_green_y, backend="jax")
#         e2_2_2 = e2_2_2 + 4 * ( -oe.contract("pr,pr->", CG, t2_green_y, backend="jax")
#                                 + 0.5 * oe.contract("iqr,qr->", GR, t2_green_y, backend="jax") )

#         # --- two-body pieces for this y (all cholesky, single tau) ---
#         # Coulomb: l2t2_c = sum_g (glgp . t)^2
#         A = oe.contract("gia,ia->g", glgp, t, backend="jax")               # (G,)
#         l2t2_c = oe.contract("g,g->", A, A, backend="jax")
#         # Exchange via the small v x v intermediate (v < o here)
#         P = oe.contract("gib,ia->gab", glgp, t, backend="jax")             # (G,v,v)
#         l2t2_e = oe.contract("gab,gba->", P, P, backend="jax")
#         e2_2_3 = e2_2_3 + 2 * l2t2_c - l2t2_e

#         return (t2o, e1_2_2, e2_2_2, e2_2_3), None

#     init = (jnp.zeros((), ctype),) * 4
#     (t2o, e1_2_2, e2_2_2, e2_2_3), _ = lax.scan(y_step, init, tau)

#     # ---- assemble ----
#     e1_2 = e1_0 * t2o + e1_2_2          # <psi|T2 h1|phi>/<psi|phi>
#     e2_2 = e2_0 * t2o + e2_2_2 + e2_2_3
#     e0 = e1_0 + e2_0                    # <psi|(h1+h2)|phi>/<psi|phi>
#     e1 = e1_2 + e2_2                    # <psi|T2(h1+h2)|phi>/<psi|phi>
#     return obar, t2o, e0, e1

In [60]:
from afqmc import t2_tools
wave_data["tau"] = t2_tools.decompose_rt2(wave_data["t2"], 1e-6)
print(wave_data["tau"].shape)
print(f"rank reduction: "
      f"{wave_data['t2'].shape[0]*wave_data['t2'].shape[1]}"
      f" -> {wave_data['tau'].shape[0]}")

(515, 5, 103)
rank reduction: 515 -> 515


In [61]:
t2_rec = oe.contract('gia,gjb->iajb', wave_data["tau"], wave_data["tau"], backend='jax')
print(abs(t2_rec - wave_data["t2"]).max())

1.0977546996415732e-16


In [79]:
walker = jnp.array(np.random.rand(*walker_init.shape))

obar, t2, e0, e1 = \
    trial._calc_energy_pt(walker, ham_data, wave_data)
print(pt2_energy_formula(h0, t2, e0, e1))

obar, t2, e0, e1 = \
    _calc_energy_pt2_decomposed(trial, walker, ham_data, wave_data)
print(pt2_energy_formula(h0, t2, e0, e1))

# obar, t2, e0, e1 = \
#     _calc_energy_pt2_decomposed_new(trial, walker, ham_data, wave_data)
# print(pt2_energy_formula(h0, t2, e0, e1))

(747.6748943387865+0j)
(747.6748943387765+0j)


In [81]:
import time
import numpy as np

try:
    import jax
    _HAS_JAX = True
except ImportError:
    _HAS_JAX = False


def benchmark(fn, n_warmup=3, n_runs=30, label=""):
    """Time a zero-arg callable. Handles JAX JIT warm-up + async dispatch."""
    # Warm-up: triggers JIT compile; NOT timed
    for _ in range(n_warmup):
        out = fn()
        if _HAS_JAX:
            jax.block_until_ready(out)

    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        out = fn()
        if _HAS_JAX:
            jax.block_until_ready(out)   # wait for the real compute to finish
        times.append(time.perf_counter() - t0)

    t = np.asarray(times)
    print(f"{label:20s} {t.mean()*1e3:8.3f} ± {t.std()*1e3:6.3f} ms "
          f"(min {t.min()*1e3:8.3f} ms, n={n_runs})")
    return t.mean(), t.min()


# the two calls as zero-arg thunks
old = lambda: trial._calc_energy_pt(walker, ham_data, wave_data)
new = lambda: _calc_energy_pt2_decomposed(trial, walker, ham_data, wave_data)
# new = lambda: _calc_energy_pt2_decomposed(trial, walker, ham_data, wave_data)

# sanity: confirm they still agree before timing
o1, o2 = old(), new()
e_old = pt2_energy_formula(h0, o1[1], o1[2], o1[3])
e_new = pt2_energy_formula(h0, o2[1], o2[2], o2[3])
print(f"energy old={e_old}, new={e_new}, diff={abs(e_old - e_new):.3e}\n")

# benchmark
mean_old, min_old = benchmark(old, label="old (dense T2):")
mean_new, min_new = benchmark(new, label="new (decomposed):")

print(f"\nspeedup (mean): {mean_old/mean_new:.2f}x")
print(f"speedup (min):  {min_old/min_new:.2f}x")

energy old=(747.6748943387865+0j), new=(747.6748943387765+0j), diff=1.000e-11

old (dense T2):        13.591 ±  0.204 ms (min   13.526 ms, n=30)
new (decomposed):     117.992 ±  0.045 ms (min  117.945 ms, n=30)

speedup (mean): 0.12x
speedup (min):  0.11x
